In [ ]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import json

In [ ]:
def create_visualizations():
    print("Генерація візуалізацій...")
    
    # 1. Читаємо шляхи з Docker
    db_path = os.getenv('DB_PATH', '/db/analytics.db')
    plots_dir = os.getenv('PLOTS_DIR', '/plots')
    
    # 2. Беремо чисті дані з БД
    conn = sqlite3.connect(db_path)
    df = pd.read_sql("SELECT * FROM inspections", conn)
    conn.close()

    os.makedirs(plots_dir, exist_ok=True)
    generated_plots = []

    # ==========================================
    # ГРАФІК 1: Топ-10 регіонів
    # ==========================================
    target_col = 'Кількість складених матеріалів з ознаками кримінальних правопорушень всього'
    
    # Страхуємось від NaN (фікс твоєї старої помилки!)
    df_clean = df.dropna(subset=['Назва регіону', target_col]).copy()
    df_clean['Назва регіону'] = df_clean['Назва регіону'].astype(str)

    top_10 = df_clean.nlargest(10, target_col).sort_values(target_col, ascending=True)

    plt.figure(figsize=(10, 6))
    bars = plt.barh(top_10['Назва регіону'], top_10[target_col], color='skyblue')
    plt.title('Топ-10 регіонів за кількістю виявлених правопорушень', fontsize=14)
    plt.xlabel('Кількість матеріалів (од.)')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    
    for bar in bars:
        plt.text(bar.get_width(), bar.get_y() + bar.get_height()/2, f' {int(bar.get_width())}', va='center')

    plot1_path = os.path.join(plots_dir, 'top_regions_chart.png')
    plt.savefig(plot1_path, bbox_inches='tight')
    plt.close()
    generated_plots.append('top_regions_chart.png')

    # ==========================================
    # ГРАФІК 2: Планові vs Позапланові
    # ==========================================
    planned = df_clean[' у т.ч. планових'].sum()
    unplanned = df_clean['у т.ч. позапланових'].sum()
    
    plt.figure(figsize=(6, 5))
    plt.bar(['Планові', 'Позапланові'], [planned, unplanned], color=['#34d399', '#f59e0b'])
    plt.title('Співвідношення перевірок')
    plt.ylabel('Кількість')
    
    plot2_path = os.path.join(plots_dir, 'planned_vs_unplanned.png')
    plt.savefig(plot2_path, bbox_inches='tight')
    plt.close()
    generated_plots.append('planned_vs_unplanned.png')

    # ==========================================
    # 3. СТВОРЕННЯ МАНІФЕСТУ ДЛЯ FLASK
    # ==========================================
    manifest_path = os.path.join(plots_dir, 'manifest.json')
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump({"plots": generated_plots}, f, ensure_ascii=False, indent=2)
        
    print(f"✅ Всі графіки та маніфест успішно збережено у: {plots_dir}")

In [ ]:
create_visualizations()